# CICCADA Data Calc Write Stage 2: Conformance Table Builders

Builds `conformance_voltvar_v2`, `conformance_voltwatt_v2`, `conformance_voltwattghi_v2`
from `ts` + `meta_up23c` + `all_uncurtailedpv_v2` (Stage 1 output).

**Run the cells in order.** Sections 1–4 are the smoke test on a single month /
single site-slice; do not skip to Section 5 until Section 4 comes back clean.

| Issue | Fixed in |
|---|---|
| R1 max(voltage) | `stage2_common.site_agg_cte` |
| R2 flex_export_detected = False | `stage2_common.meta_filter` (`exclude_flex=True`) |
| R3 / R9 AEST dates | `stage2_common.aest_month_window` + `temporal_cols` |
| R4 column naming | `build_conformance_voltvar` (thresholds unchanged) |
| R5 reduced non-conformance | emitted **both ways**, pending Baran |
| R7 V-VAr curtailment zone | `build_conformance_voltvar` |
| R10 capability on S_99 | `as4777_curves.q_cap_absorbing_sql('P_kW','S_99')` |
| R11 curve keystone | both builders import, none re-implement |
| R12 AEST day/night | `stage2_common.temporal_cols` |
| R13 total_count | `build_conformance_voltwatt` (both tables agree) |
| R14 null_uncurtailed_P_count | both GHI-joined tables |
| R15 NULL not 0 | all curtailment columns |
| R16 2024 + 2025 | Section 5 |


## 0. Setup

In [8]:
import sys, os, time
from pathlib import Path

# --- point Python at shared/ and at this stage2 folder --------------------
ROOT = Path.cwd().parents[1]          # .../ciccada_analysis
sys.path.insert(0, str(ROOT / "shared"))
sys.path.insert(0, str(ROOT / "data_calc_write" / "stage2_conformance"))

from aws_config import aq, tables, databases
from ciccada_config import SA, SAI

import build_conformance_voltvar as vv
import build_conformance_voltwatt as vw
from stage2_common import aest_month_window

DB = SAI          # solar_analytics_iceberg
N_PARTS = 8       # site slices: site_id % N_PARTS

print("targets:","\n", vv.TARGET, "\n", vw.TARGET_BASIC,"\n", vw.TARGET_GHI)

targets: 
 conformance_voltvar_v2 
 conformance_voltwatt_v2 
 conformance_voltwattghi_v2


In [9]:
# Connection + Stage 1 dependency check.
# Iceberg tables return nothing from DESCRIBE -- use SELECT * LIMIT 1.
aq("SELECT count(*) AS n_rows, count(DISTINCT site_id) AS n_sites "
   "FROM all_uncurtailedpv_v2", database=DB)

,n_rows,n_sites
0,884408066,14229


## 1. Read the SQL before running it


`preview_sql` builds the exact INSERT for one slice without executing it.


Check the AEST window: for AEST January 2024 it should read UTC partitions
`(2023,12)` and `(2024,1)`, from `2023-12-31 14:00:00` to `2024-01-31 14:00:00`.

In [10]:
print(aest_month_window(2024, 1))   # -> ('2023-12-31 14:00:00', '2024-01-31 14:00:00', [(2023,12),(2024,1)])
print(aest_month_window(2024, 7))
print(aest_month_window(2025, 12))

('2023-12-31 14:00:00', '2024-01-31 14:00:00', [(2023, 12), (2024, 1)])
('2024-06-30 14:00:00', '2024-07-31 14:00:00', [(2024, 6), (2024, 7)])
('2025-11-30 14:00:00', '2025-12-31 14:00:00', [(2025, 11), (2025, 12)])


In [11]:
print(vv.preview_sql(year=2024, month=1, n_parts=N_PARTS, part=0)[:4000])


    INSERT INTO conformance_voltvar_v2
    WITH
    
    data AS (
        SELECT
            m.site_id,
            ts.t_stamp,
            sum(ts.power * m.circuit_polarity) / 1000 AS P_kW,
            sum(ts.energy_reactive * m.circuit_polarity) / 1000 * 12 AS Q_kvar,
            max(ts.voltage)        AS V,               -- R1: was avg(voltage)
            max(m.ac_capacity_kw)  AS ac_capacity_kw,  -- nameplate: drives the STANDARD's curves
            max(m.s_99)            AS S_99             -- empirical: drives the CAPABILITY curve (R10)
        FROM ts
        JOIN (
            SELECT DISTINCT site_id, circuit_id, circuit_polarity, ac_capacity_kw, s_99
            FROM meta_up23c
            WHERE is_pv = True AND ac_capacity_kw > 0 AND s_99 > 0 AND flex_export_detected = False AND site_id % 8 = 0
        ) AS m ON ts.circuit_id = m.circuit_id
        WHERE ((ts.year = 2023 AND ts.month = 12) OR (ts.year = 2024 AND ts.month = 1))
          AND ts.t_stamp >= TIMESTAMP '2023-1

## 2. Create the empty tables

Destructive: drops and recreates. `_v2` suffix throughout.
(originals never touched)

In [ ]:
'''
print(vv.create_table(aq, database=DB))
print(vw.create_table_basic(aq, database=DB))
print(vw.create_table_ghi(aq, database=DB))
time.sleep(5)   # Glue catalog is eventually consistent
tables(DB)[tables(DB)["Table"].str.contains("conformance")][["Table"]]
'''

Created empty conformance_voltvar_v2
Created empty conformance_voltwatt_v2
Created empty conformance_voltwattghi_v2


,Table
3,conformance_antiisland
4,conformance_sust_op
5,conformance_sust_op_3w
6,conformance_voltvar
7,conformance_voltvar_v2
8,conformance_voltwatt
9,conformance_voltwatt_v2
10,conformance_voltwattghi
11,conformance_voltwattghi_v2
15,review_conformance_sust_op


In [ ]:
print(vv.create_table(aq, database=DB))       # drops + recreates conformance_voltvar_v2
print(vw.create_table_ghi(aq, database=DB))   # drops + recreates conformance_voltwattghi_v2
# do NOT recreate conformance_voltwatt_v2 — it's clean

In [ ]:
vv.run_months_voltvar(aq, database=DB, year=2024, months=list(range(1, 13)), n_parts=N_PARTS)
vv.run_months_voltvar(aq, database=DB, year=2025, months=list(range(1, 13)), n_parts=N_PARTS)

In [ ]:
vw.run_months_ghi(aq, database=DB, year=2024, months=list(range(1, 13)), n_parts=N_PARTS)
vw.run_months_ghi(aq, database=DB, year=2025, months=list(range(1, 13)), n_parts=N_PARTS)

In [ ]:
vv.validate(aq, database=DB)
vw.validate_ghi(aq, database=DB)

In [ ]:
vw.cross_check(aq, database=DB)

## 3. Test one AEST month, one site slice

`parts=[0]` is 1/8 of sites for AEST January 2024. This should complete in a few minutes. 

In [13]:
vv.run_months_voltvar(aq, database=DB, year=2024, months=[1],
                      n_parts=N_PARTS, parts=[0], exclude_flex=True)

loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])


['loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])']

In [14]:
vw.run_months_basic(aq, database=DB, year=2024, months=[1],
                    n_parts=N_PARTS, parts=[0], exclude_flex=True)
vw.run_months_ghi(aq, database=DB, year=2024, months=[1],
                  n_parts=N_PARTS, parts=[0], exclude_flex=True)

loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])


['loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])']

## 4. Validate the test

Every "MUST be 0" line must actually be 0 before you go any further.

The one to watch is **duplicate keys**. If that is non-zero, the AEST window logic has leaked and a site-day has been split across two INSERTs.

In [15]:
vv.validate(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1   83073     1418      31

Duplicate (year,month,day,day_night,site_id) keys (MUST be 0): 0

Coherence (all MUST be 0):
 neg_curtailment_rows  bucket_mismatch_rows  count_inversion_rows
                    0                     0                     0

Counterfactual coverage in the V-VAr curtailment zone:
 eligible  no_counterfactual  pct_missing
     4437               3717        83.77


In [16]:
vw.validate_basic(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1   83073     1418      31

Duplicate keys (MUST be 0): 0

Coherence (all MUST be 0):
 neg_rows  count_inversion_rows  impossible_rows
        0                     0                0


In [17]:
vw.validate_ghi(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1   83073     1418      31

Duplicate keys (MUST be 0): 0

Coherence (all MUST be 0):
 neg_curtailment_rows  impossible_rows  count_inversion_rows
                    0                0                     0

Counterfactual coverage above 253 V (R14):
 exposed_intervals  no_counterfactual  pct_missing
             49215              28095        57.09


In [18]:
# Eyeball actual rows. 
# Check: day spans 1..31, day_night has both values,
# and P_kW_sum is positive during the day.
aq(f"""
    SELECT site_id, year, month, day, day_night,
           round(P_kW_sum, 1)                   AS P_kW_sum,
           round(nonconformance_voltvar_sum, 3) AS nonconf,
           round(curtailment_voltvar_sum, 3)    AS curtail,
           curtailment_eligible_count, null_uncurtailed_P_count,
           exposed_count, all_intervals_count, total_count
    FROM {vv.TARGET}
    ORDER BY curtailment_voltvar_sum DESC NULLS LAST
    LIMIT 15
""", database=DB)

,site_id,year,month,day,day_night,P_kW_sum,nonconf,curtail,curtailment_eligible_count,null_uncurtailed_P_count,exposed_count,all_intervals_count,total_count
0,616600992,2024,1,18,day,21846.3,584.395,17.712,32,0,267,267,267
1,616600992,2024,1,19,day,19631.2,560.972,15.503,9,3,247,247,247
2,46014256,2024,1,2,day,837.9,17.563,3.481,2,0,187,228,228
3,1147413752,2024,1,19,day,13443.9,7.088,2.648,3,1,95,255,255
4,476487840,2024,1,29,day,5105.4,38.545,2.411,20,2,251,251,251
5,375244288,2024,1,9,day,951.1,0.000,1.426,2,0,192,255,255
6,114611480,2024,1,18,day,2457.7,44.685,1.315,8,2,251,251,251
7,672712144,2024,1,24,day,785.8,285.778,1.240,8,0,88,247,247
8,1345392728,2024,1,12,day,578.3,0.945,1.064,6,0,200,200,200
9,234051848,2024,1,13,day,1060.5,2.498,1.018,11,7,122,204,204


In [19]:
# regression test: the AEST boundary.
# Under original UTC extraction, intervals from 00:00-09:55 AEST were booked to
# the previous day. Here, day 1 of the month must contain a full AEST day --
# including its early-morning (night) intervals, which live in the PREVIOUS UTC
# month's partition. If day=1 has far fewer intervals than day=2, the window
# logic is wrong.
# NOTE: Since the data starts on 2024, we are missing the 10h from 31-DEC-2023, so day=1 will have 10 fewer intervals than day=2.
aq(f"""
    SELECT day, sum(all_intervals_count) AS intervals, count(DISTINCT site_id) AS sites
    FROM {vv.TARGET}
    WHERE year = 2024 AND month = 1 AND day IN (1, 2, 15, 30, 31)
    GROUP BY day ORDER BY day
""", database=DB)

,day,intervals,sites
0,1,272872,1330
1,2,454272,1333
2,15,465011,1346
3,30,438037,1368
4,31,425920,1368


## 5. Full load (2024 **and** 2025)

Each call is one AEST month * 8 site-slices = 8 Athena queries. 
A full year is 96 queries per table. 
Run one table at a time and check the printout.

If Athena throttles (`TooManyRequestsException`), 
drop `N_PARTS` to 4 or run `months` in two halves.

In [20]:
# Volt-VAr, 2024. Part 0 of Jan is already loaded from above
# rerun it and you WILL double-count. Load Jan parts 1..7:
vv.run_months_voltvar(aq, database=DB, year=2024, months=[1],
                      n_parts=N_PARTS, parts=list(range(1, N_PARTS)))
# then Feb-Dec fully:
vv.run_months_voltvar(aq, database=DB, year=2024, months=list(range(2, 13)),
                      n_parts=N_PARTS)

loaded AEST 2024-01 part=1/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=2/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=3/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=4/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=5/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=6/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=7/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-02 part=0/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions [(2024, 1), (2024, 2)])
loaded AEST 2024-02 part=1/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions 

['loaded AEST 2024-02 part=0/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions [(2024, 1), (2024, 2)])',
 'loaded AEST 2024-02 part=1/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions [(2024, 1), (2024, 2)])',
 'loaded AEST 2024-02 part=2/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions [(2024, 1), (2024, 2)])',
 'loaded AEST 2024-02 part=3/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions [(2024, 1), (2024, 2)])',
 'loaded AEST 2024-02 part=4/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions [(2024, 1), (2024, 2)])',
 'loaded AEST 2024-02 part=5/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions [(2024, 1), (2024, 2)])',
 'loaded AEST 2024-02 part=6/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions [(2024, 1), (2024, 2)])',
 'loaded AEST 2024-02 part=7/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions [(2024, 1), (2024, 2)])',
 'loaded AEST 2024-03 part=0/8 (UTC 2024-02-29 14:00:00 -> 2024-

In [21]:
vv.run_months_voltvar(aq, database=DB, year=2025, months=list(range(1, 13)),
                      n_parts=N_PARTS)

loaded AEST 2025-01 part=0/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])
loaded AEST 2025-01 part=1/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])
loaded AEST 2025-01 part=2/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])
loaded AEST 2025-01 part=3/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])
loaded AEST 2025-01 part=4/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])
loaded AEST 2025-01 part=5/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])
loaded AEST 2025-01 part=6/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])
loaded AEST 2025-01 part=7/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])
loaded AEST 2025-02 part=0/8 (UTC 2025-01-31 14:00:00 -> 2025-02-28 14:00:00, partitions

['loaded AEST 2025-01 part=0/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=1/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=2/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=3/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=4/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=5/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=6/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=7/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-02 part=0/8 (UTC 2025-01-31 14:00:00 

In [22]:
# Volt-Watt basic -- same pattern (Jan part 0 already loaded)
vw.run_months_basic(aq, database=DB, year=2024, months=[1],
                    n_parts=N_PARTS, parts=list(range(1, N_PARTS)))
vw.run_months_basic(aq, database=DB, year=2024, months=list(range(2, 13)), n_parts=N_PARTS)
vw.run_months_basic(aq, database=DB, year=2025, months=list(range(1, 13)), n_parts=N_PARTS)

loaded AEST 2024-01 part=1/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=2/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=3/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=4/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=5/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=6/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=7/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-02 part=0/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions [(2024, 1), (2024, 2)])
loaded AEST 2024-02 part=1/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions 

['loaded AEST 2025-01 part=0/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=1/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=2/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=3/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=4/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=5/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=6/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=7/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-02 part=0/8 (UTC 2025-01-31 14:00:00 

In [23]:
# Volt-Watt GHI -- same pattern (Jan part 0 already loaded)
vw.run_months_ghi(aq, database=DB, year=2024, months=[1],
                  n_parts=N_PARTS, parts=list(range(1, N_PARTS)))
vw.run_months_ghi(aq, database=DB, year=2024, months=list(range(2, 13)), n_parts=N_PARTS)
vw.run_months_ghi(aq, database=DB, year=2025, months=list(range(1, 13)), n_parts=N_PARTS)

loaded AEST 2024-01 part=1/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=2/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=3/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=4/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=5/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=6/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=7/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-02 part=0/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions [(2024, 1), (2024, 2)])
loaded AEST 2024-02 part=1/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions 

['loaded AEST 2025-01 part=0/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=1/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=2/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=3/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=4/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=5/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=6/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-01 part=7/8 (UTC 2024-12-31 14:00:00 -> 2025-01-31 14:00:00, partitions [(2024, 12), (2025, 1)])',
 'loaded AEST 2025-02 part=0/8 (UTC 2025-01-31 14:00:00 

## 6. Full validation

In [24]:
vv.validate(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1  667804    11339      31
 2024      2  641181    11540      29
 2024      3  692436    11522      31
 2024      4  675848    11631      30
 2024      5  698418    11733      31
 2024      6  668280    11675      30
 2024      7  662586    11303      31
 2024      8  657899    11068      31
 2024      9  635615    11019      30
 2024     10  657154    11072      31
 2024     11  640374    11158      30
 2024     12  670444    11165      31
 2025      1  671797    11357      31
 2025      2  598275    11123      28
 2025      3  633925    10769      31
 2025      4  614536    10552      30
 2025      5  628434    10542      31
 2025      6  517058    10269      30
 2025      7  618363    10458      31
 2025      8  606885    10147      31
 2025      9  573473     9975      30
 2025     10  579826     9766      31
 2025     11  544404     9463      30
 2025     12  546209     9183      31

Duplicate (ye

In [25]:
vw.validate_basic(aq, database=DB);
vw.validate_ghi(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1  667804    11339      31
 2024      2  641181    11540      29
 2024      3  692436    11522      31
 2024      4  675848    11631      30
 2024      5  698418    11733      31
 2024      6  668280    11675      30
 2024      7  662586    11303      31
 2024      8  657899    11068      31
 2024      9  635615    11019      30
 2024     10  657154    11072      31
 2024     11  640374    11158      30
 2024     12  670444    11165      31
 2025      1  671797    11357      31
 2025      2  598275    11123      28
 2025      3  633925    10769      31
 2025      4  614536    10552      30
 2025      5  628434    10542      31
 2025      6  517058    10269      30
 2025      7  618363    10458      31
 2025      8  606885    10147      31
 2025      9  573473     9975      30
 2025     10  579826     9766      31
 2025     11  544404     9463      30
 2025     12  546209     9183      31

Duplicate key

In [26]:
# R13 regression test: the two Volt-Watt tables must share a denominator.
vw.cross_check(aq, database=DB);

Site-days where basic.total_count != ghi.total_count (MUST be 0): 192996


In [33]:
# If no duplicates in the source, compare the tables directly
aq(f"""
    SELECT b.total_count AS basic_tc, g.total_count AS ghi_tc,
           g.total_count - b.total_count AS diff,
           b.all_intervals_count AS basic_all, g.all_intervals_count AS ghi_all
    FROM {vw.TARGET_BASIC} b
    JOIN {vw.TARGET_GHI} g
      ON b.site_id = g.site_id AND b.year = g.year
     AND b.month = g.month AND b.day = g.day AND b.day_night = g.day_night
    WHERE b.total_count <> g.total_count
    LIMIT 10
""", database=DB)

,basic_tc,ghi_tc,diff,basic_all,ghi_all
0,18,36,18,144,230
1,7,12,5,95,165
2,39,42,3,144,155
3,4,6,2,142,189
4,2,4,2,144,225
5,5,9,4,144,213
6,10,18,8,143,221
7,3,6,3,143,224
8,72,80,8,144,168
9,2,3,1,144,204


## 7. Reconcile against original tables

Differences are **expected**. They should be fully explained by:

- flex-export sites excluded (biggest effect; ~700 sites in Stage 1)
- `max(voltage)` is >= `avg(voltage)`, so more intervals cross 240 V / 253 V
- AEST day boundaries reshuffle intervals between days
- capability clamped on `s_99`, not nameplate

If the site-count drop does **not** match the flex-export count, stop and find out why before trusting anything.

In [27]:
vv.compare_to_original(aq, database=DB, original="conformance_voltvar");

sites in conformance_voltvar_v2:  15,609
sites in conformance_voltvar: 16,147
sites dropped:       539
flex-export sites (expected explanation for the drop): 539


In [28]:
# Fleet-level headline comparison. Expect the same order of magnitude, not
# identical numbers.
aq(f"""
    SELECT 'v2' AS tbl, year,
           round(sum(nonconformance_voltvar_sum), 0)  AS nonconf_kvar,
           round(sum(curtailment_voltvar_sum), 0)     AS curtail_kw,
           sum(total_count)                           AS intervals
    FROM {vv.TARGET} GROUP BY year
    ORDER BY year
""", database=DB)

,tbl,year,nonconf_kvar,curtail_kw,intervals
0,v2,2024,411516800.0,2235.0,1359018262
1,v2,2025,336738298.0,922.0,1203011227


## 8. the number you cannot publish yet

`nonconformance_voltvar_red_sum` reproduces Hossein's definition:
`adverse + inactive + near_conformant` — i.e. it counts the sites that are
essentially complying and ignores the ones falling well short.

`nonconformance_voltvar_red_alt_sum` is `adverse + inactive + significant_shortfall`
— what the definition almost certainly should be.

Run this, take both numbers to Baran, and get him to confirm which range CANVAS
intended before either appears in the paper.

In [30]:
aq(f"""
    SELECT year,
           round(sum(nonconformance_voltvar_red_sum), 0)     AS red_hossein,
           round(sum(nonconformance_voltvar_red_alt_sum), 0) AS red_alt,
           sum(nonconformance_voltvar_red_count)             AS red_hossein_count,
           sum(nonconformance_voltvar_red_alt_count)         AS red_alt_count,
           round(sum(Q_near_conformant_sum), 0)              AS near_conformant,
           round(sum(Q_significant_shortfall_sum), 0)        AS significant_shortfall
    FROM {vv.TARGET}
    GROUP BY year ORDER BY year
""", database=DB)

,year,red_hossein,red_alt,red_hossein_count,red_alt_count,near_conformant,significant_shortfall
0,2024,360891506.0,386023761.0,295789798,318288620,641671.0,25773925.0
1,2025,292079479.0,316248498.0,243475804,266923371,572945.0,24741963.0


In [32]:
aq("""
    SELECT count(*) AS n_duplicate_keys
    FROM (
        SELECT site_id, t_stamp
        FROM all_uncurtailedpv_v2
        GROUP BY site_id, t_stamp
        HAVING count(*) > 1
    )
""", database=DB)

,n_duplicate_keys
0,418963867
